# Social Media Trend Analysis
### Team Project Semester 4
**Task:** Reddit Data Collection for Apple, Tesla, and Amazon 
**Subreddits:** r/stocks, r/investing, r/stockmarket 
**Author:** Percival - Project Coordinator 
**Date:** March 27, 2026

---
### Notebook Overview
This notebook documents the full data collection process for the Social Media Trend Analysis task. It pulls Reddit post data for Apple (AAPL), Tesla (TSLA), and Amazon (AMZN) across three subreddits, cleans the dataset, and produces an initial summary of engagement metrics. The cleaned dataset feeds directly into the integration with Shamil's stock market data in the next milestone.

## Cell 1: Install Required Libraries

In [ ]:
!pip install requests pandas

**Result:** All required libraries were already installed on this machine. requests version 2.32.3 and pandas version 2.2.2 are both up to date and ready to use. No additional installation was needed.

## Cell 2: Import Libraries

In [ ]:
import requests
import pandas as pd
from datetime import datetime
import time

print('Libraries imported successfully.')

**Result:** All four libraries imported without errors. requests handles the API calls to Reddit, pandas manages the dataframe, datetime converts Unix timestamps to readable dates, and time manages the pause between requests to avoid rate limiting.

## Cell 3: Define Companies and Subreddits

In [ ]:
companies = {
    "Apple": "AAPL",
    "Tesla": "TSLA",
    "Amazon": "AMZN"
}

subreddits = ["stocks", "investing", "stockmarket"]
headers = {"User-Agent": "team-project-semester4/0.1"}

print(f'Tracking {len(companies)} companies across {len(subreddits)} subreddits.')
print(f'Companies: {list(companies.keys())}')
print(f'Subreddits: {subreddits}')

**Result:** Configuration confirmed. The script is set to track 3 companies across 3 subreddits, giving a maximum possible pull of 9 batches of 100 posts each (900 total raw posts). These companies were agreed with Shamil as the shared anchor for the stock market integration. The same tickers will be used in his yfinance data pull.

## Cell 4: Pull Reddit Data

In [ ]:
posts = []

for subreddit in subreddits:
    for company, ticker in companies.items():
        url = f"https://www.reddit.com/r/{subreddit}/search.json"
        params = {
            "q": ticker,
            "t": "year",
            "limit": 100,
            "sort": "relevance"
        }
        response = requests.get(url, headers=headers, params=params)
        
        if response.status_code == 200:
            data = response.json()
            for post in data["data"]["children"]:
                p = post["data"]
                posts.append({
                    "company": company,
                    "ticker": ticker,
                    "subreddit": subreddit,
                    "title": p["title"],
                    "score": p["score"],
                    "upvote_ratio": p["upvote_ratio"],
                    "num_comments": p["num_comments"],
                    "date": datetime.utcfromtimestamp(
                        p["created_utc"]
                    ).strftime("%Y-%m-%d"),
                    "url": p["url"]
                })
            print(f'Pulled {company} ({ticker}) from r/{subreddit} - {len(data["data"]["children"])} posts')
        else:
            print(f'Failed: r/{subreddit} - {company} - Status {response.status_code}')
        
        time.sleep(2)

print(f'\nTotal raw posts collected: {len(posts)}')

**Result:** All 9 batches returned exactly 100 posts each, confirming that all three subreddits responded successfully with no failed requests. A total of 900 raw posts were collected.

One deprecation warning appeared for datetime.utcfromtimestamp(). This does not affect the output but will be fixed in the next version of this notebook by switching to timezone-aware datetime objects.

The consistent 100 posts per batch suggests Reddit capped results at 100 per search query, which is the default limit of the JSON API. This is expected behavior and is sufficient for our analysis.

## Cell 5: Convert to DataFrame and Remove Duplicates

In [ ]:
df = pd.DataFrame(posts)
df = df.drop_duplicates(subset="url")
df = df.sort_values("date").reset_index(drop=True)

print(f'Total posts after removing duplicates: {len(df)}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'\nPosts per company:')
print(df["company"].value_counts())
print(f'\nPosts per subreddit:')
print(df["subreddit"].value_counts())

**Result:** After removing duplicates, 620 posts were dropped, leaving 280 unique posts. This is a significant reduction from 900 to 280, meaning roughly 69% of the raw posts were duplicates appearing across multiple subreddits.

Key observations:

- The date range spans March 28, 2025 to March 26, 2026, giving us almost exactly 12 months of data. This aligns perfectly with Shamil's stock data pull period.
- Apple has the most unique posts (100), followed by Tesla (92) and Amazon (88). Apple generates slightly more distinct Reddit discussion than the other two.
- After deduplication, all 280 posts fall under r/stocks. This tells us that the most relevant and unique posts for these three tickers are concentrated in r/stocks. The posts from r/investing and r/stockmarket were largely cross-posts of the same content.

This is useful insight for future data pulls. Focusing on r/stocks alone for these tickers would be sufficient.

## Cell 6: Preview the Dataset

In [ ]:
print(f'Dataset shape: {df.shape}')
print(f'\nColumn names: {list(df.columns)}')
print(f'\nFirst 5 rows:')
df.head()

**Result:** The dataset has 280 rows and 9 columns. All required columns are present: company, ticker, subreddit, title, score, upvote_ratio, num_comments, date, and url.

Looking at the first 5 rows, several things stand out immediately:

- The earliest posts from March 28, 2025 include a high-engagement Apple post about over 1.25 trillion dollars wiped from the US stock market, scoring 4421 upvotes with 402 comments. This likely corresponds to a significant market event worth cross-referencing with Shamil's stock data for the same date.
- A Tesla post about Elon Musk's plans to leave DOGE scored 1630 upvotes with 434 comments on the same date, showing that non-financial news about leadership also drives strong Reddit engagement.
- An Amazon post titled "Smile everyone!" scored the highest in the preview at 23775 upvotes with 563 comments on March 31, 2025. This unusually high score warrants investigation during the analysis phase.
- Upvote ratios across all 5 rows are between 0.90 and 0.98, indicating these posts had strong community agreement with minimal downvoting.

## Cell 7: Check for Missing Values

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nData types:')
print(df.dtypes)

**Result:** Zero missing values across all 9 columns. The dataset is complete with no gaps to fill or rows to drop.

On data types, two things need attention in the next notebook:

- The date column is stored as object (string) rather than datetime. It needs to be converted to datetime format before any time-series analysis or weekly aggregation.
- score and num_comments are correctly stored as int64. upvote_ratio is correctly float64. These are ready for numerical analysis as-is.

## Cell 8: Save to CSV

In [ ]:
df.to_csv("reddit_data.csv", index=False)
print(f'Dataset saved to reddit_data.csv')
print(f'Total posts saved: {len(df)}')

**Result:** 280 unique posts saved successfully to reddit_data.csv. This file is the input dataset for all subsequent notebooks in this task. Push this file to GitHub under /social-media-trends/ before the end of today.

## Cell 9: Quick Summary Stats

In [ ]:
print('Engagement summary by company:')
df.groupby('company')[['score', 'upvote_ratio', 'num_comments']].mean().round(2)

**Result:** The engagement summary reveals clear differences between the three companies:

Apple leads on average score (4596.38), meaning Apple posts receive the most upvotes on average. This suggests Apple content generates broader community interest and approval on Reddit.

Amazon leads on upvote ratio (0.91), meaning Amazon posts face the least disagreement or downvoting relative to total votes. Amazon discussions tend to be more consensus-driven.

Tesla generates the most comments relative to its score (373.72 average comments against an average score of 2838.23). This means Tesla posts provoke more discussion and debate per upvote than Apple or Amazon. Tesla has a more polarizing community presence on Reddit, which is consistent with broader public perception of the company and its CEO.

These differences are meaningful for the integration with Shamil. High Tesla comment volume with lower upvote scores may indicate sentiment volatility around the stock, which should surface in the correlation analysis.

---
## Milestone 1 Summary: March 27, 2026

**What was completed:**
- Reddit data collected for Apple, Tesla, and Amazon across r/stocks, r/investing, and r/stockmarket
- 900 raw posts pulled, 280 unique posts retained after deduplication
- Date range: March 28, 2025 to March 26, 2026 (12 months)
- Zero missing values. Dataset is clean and ready for preprocessing.
- Dataset saved to reddit_data.csv

**Key early findings:**
- Apple generates the highest average engagement score
- Tesla generates the most comments per post, suggesting higher sentiment volatility
- Amazon posts show the strongest community consensus
- All unique posts are concentrated in r/stocks after deduplication

**Action items before Milestone 2 (April 3):**
1. Push reddit_data.csv to GitHub under /social-media-trends/
2. Notify Shamil on Teams: companies confirmed as Apple (AAPL), Tesla (TSLA), Amazon (AMZN) with date range March 2025 to March 2026
3. Convert date column from string to datetime in the cleaning notebook
4. Begin preprocessing and sentiment analysis notebook